# 02 — Deep Space Analysis: Nuclear Energy Destruction Limits

**Repository:** `sayan9168/Sayan-Universal-Gradient-Theory`  
**Zenodo DOI:** [10.5281/ZENODO.22723906](https://doi.org/10.5281/ZENODO.22723906)  
**Package:** `sayan-gradient`

---

## Overview

In deep space vacuum, energy propagates without atmospheric attenuation, leaving geometric logarithmic fall-off as the governing decay mechanism.

Postulate V states that matter transitions into an information/energy degree of freedom once $\theta_S \ge \theta_c$. Inverting the field yields:
- **Vacuum destruction radius:**
  $$R_d = r_0 \exp\!\left( \frac{K\sqrt{e}}{\tau \theta_c} \right)$$
- **Energy breaking threshold:**
  $$E_c = E_0 \left( \frac{\tau \theta_c \ln(r/r_0)}{K} \right)^2$$
- **Parameter-Free Energy Ratio (Prediction P2):**
  $$\frac{E_2}{E_1} = \left( \frac{\ln(r_2/r_0)}{\ln(r_1/r_0)} \right)^2$$

In this notebook, we analyze:
1. Scaling of vacuum destruction radius vs released energy.
2. Breaking energy threshold required to disintegrate targets at cosmic distances.
3. Empirical testing of Prediction P2.
4. Model calibration using SciPy optimization.
5. Destruction distance solving in non-uniform thermal gradients.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

from sayan_gradient import (
    DEEP_SPACE,
    ReferenceScales,
    vacuum_destruction_radius,
    energy_breaking_threshold,
    parameter_free_energy_ratio,
    calibrate_k,
    solve_destruction_distance,
    plot_vacuum_destruction_radius,
    plot_breaking_energy,
)

## 1. Vacuum Destruction Radius vs Source Energy

Because $R_d \propto \exp(K\sqrt{e}/(\tau\theta_c))$, the destruction radius grows exponentially with $\sqrt{e}$, producing an enormous reach in vacuum:

In [1]:
sc = ReferenceScales(E0=1.0, T0=1.0, r0=1.0, K=1.0)
energies = np.array([1.0, 4.0, 9.0, 16.0, 25.0])
theta_crit = 1.0
delta_T = 1.0

radii = vacuum_destruction_radius(energies, theta_crit=theta_crit, delta_T=delta_T, scales=sc)

for e, r_d in zip(energies, radii):
    print(f"Source Energy e = {e:4.1f} | Destruction Radius R_d = {r_d:10.2f} r0")

Source Energy e =  1.0 | Destruction Radius R_d =       2.72 r0
Source Energy e =  4.0 | Destruction Radius R_d =       7.39 r0
Source Energy e =  9.0 | Destruction Radius R_d =      20.09 r0
Source Energy e = 16.0 | Destruction Radius R_d =      54.60 r0
Source Energy e = 25.0 | Destruction Radius R_d =     148.41 r0


## 2. Breaking Energy Threshold Across Cosmic Distances

Computing the energy required to project destruction to distant orbital or deep space targets:

In [1]:
target_radii = np.array([10.0, 50.0, 100.0, 500.0, 1000.0])
breaking_energies = energy_breaking_threshold(target_radii, theta_crit=1.0, delta_T=1.0, scales=sc)

for r_t, eb in zip(target_radii, breaking_energies):
    print(f"Target Distance r = {r_t:6.1f} r0 | Required Breaking Energy = {eb:10.2f} E0")

Target Distance r =   10.0 r0 | Required Breaking Energy =       5.30 E0
Target Distance r =   50.0 r0 | Required Breaking Energy =      15.30 E0
Target Distance r =  100.0 r0 | Required Breaking Energy =      21.21 E0
Target Distance r =  500.0 r0 | Required Breaking Energy =      38.62 E0
Target Distance r = 1000.0 r0 | Required Breaking Energy =      47.72 E0


## 3. Testing Prediction P2 (Parameter-Free Energy Ratio)

Prediction P2 states that the energy multiplier required to expand reach from $r_1$ to $r_2$ is strictly:
$$\frac{E_2}{E_1} = \left(\frac{\ln(r_2/r_0)}{\ln(r_1/r_0)}\right)^2$$
independent of $K$ and $\theta_c$!

In [1]:
r1 = 10.0
r2 = 100.0
r3 = 1000.0

ratio_1_to_2 = parameter_free_energy_ratio(r1, r2, r0=1.0)
ratio_2_to_3 = parameter_free_energy_ratio(r2, r3, r0=1.0)

print(f"Energy multiplier to expand reach from r={r1} to r={r2}: {ratio_1_to_2:.4f}x")
print(f"Energy multiplier to expand reach from r={r2} to r={r3}: {ratio_2_to_3:.4f}x")

Energy multiplier to expand reach from r=10.0 to r=100.0: 4.0000x
Energy multiplier to expand reach from r=100.0 to r=1000.0: 2.2500x


## 4. Calibrating $K$ from Observational Data with SciPy

Given experimental measurements or simulated anchor events, we can fit $K$ using non-linear least squares (`scipy.optimize.curve_fit`):

In [1]:
# Simulated data with true K = 1.62
true_k = 1.62
true_scales = ReferenceScales(E0=1.0, T0=1.0, r0=1.0, K=true_k)

obs_energies = np.array([5.0, 12.0, 25.0, 50.0, 80.0])
obs_dTs = np.array([1.2, 1.5, 2.0, 2.5, 3.0])
obs_radii = np.array([4.0, 8.0, 15.0, 30.0, 60.0])

from sayan_gradient import theta_s
obs_thetas = theta_s(obs_energies, obs_dTs, obs_radii, scales=true_scales)

# Add synthetic measurement noise (2%)
np.random.seed(42)
noisy_thetas = obs_thetas * (1.0 + 0.02 * np.random.randn(len(obs_thetas)))

k_fit, k_err = calibrate_k(obs_energies, obs_dTs, obs_radii, noisy_thetas, scales=sc)
print(f"True K:            {true_k:.4f}")
print(f"Calibrated K:      {k_fit:.4f} +/- {k_err:.4f}")

True K:            1.6200
Calibrated K:      1.6340 +/- 0.0088


## 5. Non-Linear Thermal Gradients via Root-Finding

When the thermal gradient $\Delta T(r)$ is a spatially varying function (e.g. radiative equilibrium $\Delta T(r) = T_0 / \sqrt{r}$), we solve for the exact destruction distance using Brent's method (`scipy.optimize.root_scalar`):

In [1]:
# Spatially varying thermal profile: gradient increases with distance
def space_thermal_profile(r: float) -> float:
    return 1.0 + 0.02 * r

r_destruction = solve_destruction_distance(
    E_nb=100.0,
    theta_crit=0.8,
    thermal_profile_fn=space_thermal_profile,
    r_bracket=(1.1, 500.0),
    scales=sc
)
print(f"Destruction distance in non-uniform thermal gradient: r = {r_destruction:.2f} r0")

Destruction distance in non-uniform thermal gradient: r = 89.18 r0


## 6. Destruction Limits Visualizations

In [1]:
fig1 = plot_vacuum_destruction_radius(thresholds=[0.5, 1.0, 2.0], tau=1.0, scales=sc)
plt.show()

fig2 = plot_breaking_energy(thresholds=[0.5, 1.0, 2.0], tau=1.0, scales=sc)
plt.show()